In [ ]:
with open('../data/archive/1of2/wiki_00', 'r', encoding='utf-8') as f:
    text = f.read()

print("The length of our dataset: ", len(text))


The length of dataset:  1030267


In [2]:
# Here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"$%&'()*+,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]^_`abcdefghijklmnopqrstuvwxyz|~ £°²³´·¹ÁÅÖ×àáâãäåæçèéêìíîïðñòóôöøúüÿāēĕěīıłōūǫəɪʻʿˈΓΔΜάέίαβγδεηθικλμνορςστυφωόύابجرشلهἀἄἡἥἶ‎–—‘’“”†…€ℓ→−♆の万八大姬燕百神蓟都
207


In [4]:
# Tokenizer! (Extremely Simple Char-level Tokenizer)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [ stoi[c] for c in s] # string to integers
decode = lambda l: ''.join(itos[i] for i in l) # integers to string

print(encode("I need to eat something"))
print(decode(encode("I need to eat something")))

[40, 1, 76, 67, 67, 66, 1, 82, 77, 1, 67, 63, 82, 1, 81, 77, 75, 67, 82, 70, 71, 76, 69]
I need to eat something


In [5]:
# Let's now encode the entire text dataset and store it into a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1030267]) torch.int64
tensor([32, 78, 80, 71, 74,  0,  0, 32, 78, 80, 71, 74,  1,  8, 32, 78, 80, 14,
         9,  1, 71, 81,  1, 82, 70, 67,  1, 68, 77, 83, 80, 82, 70,  1, 75, 77,
        76, 82, 70,  1, 77, 68,  1, 82, 70, 67,  1, 87, 67, 63, 80,  1, 71, 76,
         1, 82, 70, 67,  1, 41, 83, 74, 71, 63, 76,  1, 63, 76, 66,  1, 38, 80,
        67, 69, 77, 80, 71, 63, 76,  1, 65, 63, 74, 67, 76, 66, 63, 80, 81, 12,
         1, 63, 76, 66,  1, 65, 77, 75, 67, 81,  1, 64, 67, 82, 85, 67, 67, 76,
         1, 44, 63, 80, 65, 70,  1, 63, 76, 66,  1, 44, 63, 87, 14,  1, 40, 82,
         1, 71, 81,  1, 77, 76, 67,  1, 77, 68,  1, 68, 77, 83, 80,  1, 75, 77,
        76, 82, 70, 81,  1, 82, 77,  1, 70, 63, 84, 67,  1, 19, 16,  1, 66, 63,
        87, 81, 14,  0,  0, 32, 78, 80, 71, 74,  1, 63, 74, 85, 63, 87, 81,  1,
        64, 67, 69, 71, 76, 81,  1, 77, 76,  1, 82, 70, 67,  1, 81, 63, 75, 67,
         1, 66, 63, 87,  1, 77, 68,  1, 82, 70, 67,  1, 85, 67, 67, 73,  1, 63,
      

In [ ]:
# Let's now split up the data into train and validation set
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
# We need to sample a random chunk of data for the sake of efficient (possible) training 
# The following is the size of the chunk
block_size = 4
train_data[:block_size+1]

tensor([32, 78, 80, 71, 74])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context}, the target: {target}")

When input is tensor([32]), the target: 78
When input is tensor([32, 78]), the target: 80
When input is tensor([32, 78, 80]), the target: 71
When input is tensor([32, 78, 80, 71]), the target: 74


In [ ]:
# We need to consider batch size then (a set of independent chuncks to exploit the parallelism of GPU)
torch.manual_seed(4234)
batch_size = 4 # how many independent sequences (c.f. "chunks") will we process in parallel?
block_size = 8 # what's the maximum context length for predictions? 

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # randonly pick a sample from the training set
    x = torch.stack([data[i:i+block_size] for i in ix]) # torch.stack: stacking 1-dim tensor to 2-dim tensor (literally 'stacking' them)  
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print("---")

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1, 65, 83, 80, 84, 67,  1, 63],
        [71, 81,  1, 13, 17, 14,  0,  0],
        [74,  1, 65, 77, 80, 80, 67, 65],
        [76, 63, 12,  1, 63, 76, 66,  1]])
targets:
torch.Size([4, 8])
tensor([[65, 83, 80, 84, 67,  1, 63, 82],
        [81,  1, 13, 17, 14,  0,  0, 51],
        [ 1, 65, 77, 80, 80, 67, 65, 82],
        [63, 12,  1, 63, 76, 66,  1, 82]])
---
when input is [1] the target: 65
when input is [1, 65] the target: 83
when input is [1, 65, 83] the target: 80
when input is [1, 65, 83, 80] the target: 84
when input is [1, 65, 83, 80, 84] the target: 67
when input is [1, 65, 83, 80, 84, 67] the target: 1
when input is [1, 65, 83, 80, 84, 67, 1] the target: 63
when input is [1, 65, 83, 80, 84, 67, 1, 63] the target: 82
when input is [71] the target: 81
when input is [71, 81] the target: 1
when input is [71, 81, 1] the target: 13
when input is [71, 81, 1, 13] the target: 17
when input is [71, 81, 1, 13, 17] the target: 14
when input is [71, 81, 1

In [ ]:
# Now we have a input batch for our transformer
print(xb)

tensor([[ 1, 65, 83, 80, 84, 67,  1, 63],
        [71, 81,  1, 13, 17, 14,  0,  0],
        [74,  1, 65, 77, 80, 80, 67, 65],
        [76, 63, 12,  1, 63, 76, 66,  1]])


In [ ]:
# Let's look through Bigram Language Model first
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337) # 1337 is just a convention or for the sake of reproductability on any other envs.

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):

        # idx and targets are both (B, T) tensor of integers